In [2]:
from datetime import datetime

# Define the query
query = """
SELECT * 
FROM [dbo].[successful_queries]
WHERE dimension = 'Duplicity'
AND FILE_ID > ISNULL((SELECT MAX(FILE_ID) FROM duplicity_query_output), 0);
"""

# Execute the query
cursor.execute(query)

# Fetch all results
rows = cursor.fetchall()

# Define schema
schema = StructType([
    StructField('file_id', StringType()),
    StructField('value', StringType()),
    StructField('error_id', IntegerType()),
    StructField('rule_no', StringType()),
    StructField('business_rule_definition', StringType()),
    StructField('table_name', StringType()),
    StructField('column_name', StringType()),
    StructField('business_attribute_name', StringType()),
    StructField('attribute_group', StringType()),
    StructField('severity', StringType()),
    StructField('requiredAttribute', StringType()),
    StructField('dimension', StringType()),
    StructField('total_count', IntegerType()),
    StructField('extract_date', DateType()),
])

# Get column names from cursor description
columns = [column[0] for column in cursor.description]

# Convert pyodbc.Row objects to list of tuples
rows = [tuple(row) for row in rows]

# Define schema explicitly based on the columns
schema = StructType([StructField(col, StringType(), True) for col in columns])

try:
    # Create DataFrame using StructType schema
    df_query = spark.createDataFrame(rows, schema)
    df_rec = df_query.collect()
except:
    # Create an empty DataFrame with the defined schema if no data is fetched
    df_query = spark.createDataFrame([], schema)
    df_rec = df_query.collect()

In [ ]:
# List to store DataFrames
list_df = []
c = 0
# Loop through each row in the dataframe
for row in df_rec:
    try:
        # Extract values from the row
        file_id = row["file_id"]
        table_name = row["table_name"]
        column_name = row["column_name"]
        rule_no = row["rule_no"]
        business_rule_definition = row["business_rule_definition"].replace("'", "")
        attribute_group = row["attribute_group"]
        dimension = row["dimension"]

        # Create a dictionary for the values
        d = {
            'file_id': file_id, 
            'table_name': table_name,
            'column_name': column_name,
            'rule_no': rule_no,
            'business_rule_definition': business_rule_definition,
            'attribute_group': attribute_group,
            'dimension': dimension
        }
        
        # Generate a random error_id
        error_id = random.randint(1000, 9999)

        # Construct SQL query for detecting duplicates
        query = f"""
        SELECT DISTINCT
            {d['file_id']} as file_id,
            '{d['column_name']}' as value,
            '{error_id}' AS error_id,
            '{d['rule_no']}' AS rule_no,
            '{d["business_rule_definition"]}' AS business_rule_definition,
            '{d['table_name']}' AS table_name,
            '{d['column_name']}' AS column_name, 
            concat({d['column_name']}, '') AS column_value,
            concat({d['column_name']}, '') AS business_attribute_name,
            '{d['attribute_group']}' AS attribute_group, 
            'High' AS severity, 
            'YES' AS RequiredAttribute,         
            '{d['dimension']}' AS dimension,
            COUNT(*) AS total_count,
            '{datetime.now()}' AS extract_date
        FROM {table_name} 
        GROUP BY {column_name}
        HAVING COUNT(*) > 1
        """

        # print(f"Executing query:\n{query}")

        try:
            # Execute the query
            cursor.execute(query)
            rows = cursor.fetchall()
            
            # Convert result to DataFrame
            if rows:
                columns = [column[0] for column in cursor.description]
                df_final = pd.DataFrame.from_records(rows, columns=columns)
                list_df.append(df_final)
    
        except Exception as e:
            print(f"Error executing query: {e}")

    except Exception as e:
        c = c + 1
        print(f"Error processing row {row}: {str(e)}")

In [4]:
# Combine all DataFrames into one
if list_df:
    dup_final_df = list_df[0]
    for df in list_df[1:]:
        dup_final_df = dup_final_df.union(df)


    # Save output to CSV
    dup_final_df.to_sql('duplicity_query_output', con=engine, if_exists='append', index=False)

else:
    print("No duplicate records found.")